In [5]:
from sentence_transformers import SentenceTransformer
import json
import faiss
import numpy as np

# 1. Carica il modello (lo scarica automaticamente la prima volta)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Carica i chunk di Tesla che abbiamo salvato ieri
with open("../data/chunks/tsla_10k_2025_chunks.json", "r") as f:
    data = json.load(f)
    chunks = data['chunks']

# 3. Trasforma i primi 5 chunk in vettori (per test)
embeddings = model.encode(chunks, show_progress_bar=True)

print(f"Forma del vettore: {embeddings.shape}") 
# Dovrebbe essere (5, 384) -> 5 pezzi di testo, ognuno descritto da 384 numeri.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Forma del vettore: (7497, 384)


In [6]:
dimension = embeddings.shape[1] # 384
index = faiss.IndexFlatL2(dimension) # Indice basato sulla distanza Euclidea
index.add(embeddings.astype('float32')) # FAISS vuole float32

print(f"Indice pronto con {index.ntotal} vettori.")

Indice pronto con 7497 vettori.


In [7]:
query = "What are the main risks mentioned regarding cyberattacks or data security?"

# 1. Trasformiamo la domanda in un vettore
query_vector = model.encode([query]).astype('float32')

# 2. Cerchiamo i 3 pezzi di testo più simili (k=3)
k = 3
distances, indices = index.search(query_vector, k)

print(f"Risultati per: '{query}'\n")
for i, idx in enumerate(indices[0]):
    print(f"--- RISULTATO {i+1} (Distanza: {distances[0][i]:.4f}) ---")
    print(chunks[idx][:500] + "...") # Mostriamo i primi 500 caratteri del chunk
    print("\n")

Risultati per: 'What are the main risks mentioned regarding cyberattacks or data security?'

--- RISULTATO 1 (Distanza: 0.7401) ---
computer hacking, unauthorized access, exploitation of bugs, defects and vulnerabilities, breakdowns, damage, interruptions, system malfunctions, power outages, terrorism, acts of vandalism, security breaches, security incidents, inadvertent or intentional actions by employees or other third parties, and other cyber-attacks. To the extent any security incident results in unauthorized access or damage to or acquisition, use, corruption, loss, destruction, alteration or dissemination of our data, ...


--- RISULTATO 2 (Distanza: 0.7420) ---
adverse consequences. We continue to expand our information technology systems as our operations grow, such as product data management, procurement, inventory management, production planning and execution, sales, service and logistics, dealer management, financial, tax and regulatory compliance systems. This includes the 

In [8]:
import faiss
import os

# Salva l'indice FAISS
os.makedirs("../data/embeddings", exist_ok=True)
faiss.write_index(index, "../data/embeddings/tsla_index.bin")

# Salva i metadati dei chunk (fondamentale perché FAISS non salva il testo, solo i vettori)
# Abbiamo già il file tsla_10k_2025_chunks.json, quindi siamo a posto.
print("Indice salvato correttamente!")

Indice salvato correttamente!
